# Experiment 2: balanced case corpus

Five deterministic replicas sample the same number of cases per category. The common
result pack uses the median NCD matrix and the replicate closest to that median as the
representative tree; all replica outputs remain available below the result directory.


In [1]:
from pathlib import Path
import os
import sys

import pandas as pd
from IPython.display import display

PROJECT_ROOT = Path.cwd().resolve()
while PROJECT_ROOT != PROJECT_ROOT.parent and not (PROJECT_ROOT / "pyproject.toml").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
if not (PROJECT_ROOT / "pyproject.toml").exists():
    raise RuntimeError("Run this notebook from the repository or one of its subdirectories.")
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from hypotheses.violence_against_women.scripts.experiment_common import (
    artifact_paths,
    case_execution,
    category_order_from_manifest,
    ensure_artifact_directories,
    load_artifact_manifest,
    load_category_map,
    run_damicore_experiment,
    write_common_result_artifacts,
)

CATEGORY_SET_VERSION = os.getenv("DAMICORE_CATEGORY_SET_VERSION", "v2_30")
PATHS = artifact_paths(CATEGORY_SET_VERSION)
COMMON_WORK_ROOT = PATHS.common
NORMALIZED_WORK_ROOT = PATHS.normalized
CASE_FULL_WORK_ROOT = PATHS.case_full
CASE_BALANCED_WORK_ROOT = PATHS.case_balanced
RESULTS_ROOT = PATHS.results
ensure_artifact_directories(PATHS)
manifest = load_artifact_manifest(
    COMMON_WORK_ROOT / "artifact-manifest.json",
    category_set_version=CATEGORY_SET_VERSION,
)
category_order = category_order_from_manifest(manifest)
category_map = load_category_map(COMMON_WORK_ROOT / "category-map.csv")
category_support = pd.read_csv(COMMON_WORK_ROOT / "category-support.csv")
assert category_map["category"].tolist() == category_order
assert set(category_support["category"]) == set(category_order)
assert manifest["dimension_count"] == 20

from itertools import combinations

import numpy as np

from hypotheses.violence_against_women.scripts.experiment_common import (
    CASE_SEEDS,
    plot_bias_diagnostics,
    plot_cluster_stability,
    plot_ncd_heatmap,
    plot_support,
    render_tree_artifacts,
    same_cluster_pairs,
    write_json,
)


/Users/erickpatrickbarcelos/codes/data-mining/venv/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Run the five balanced replicas


In [2]:
case_category_map = pd.read_csv(COMMON_WORK_ROOT / "case-category-map.csv")
balanced_root = RESULTS_ROOT / "case_balanced"
replicate_results = {}
replicate_tables = []

for index, _seed in enumerate(CASE_SEEDS, start=1):
    replicate = f"replicate-{index:03d}"
    replicate_map = case_category_map.loc[
        (case_category_map["regime"] == "case-balanced")
        & (case_category_map["replicate"] == replicate)
    ].set_index("category").loc[category_order].reset_index()
    support = replicate_map[["category", "case_count"]].rename(
        columns={"case_count": "support"}
    )
    bytes_by_category = replicate_map.set_index("category")["bytes"].astype(int).to_dict()
    result = run_damicore_experiment(
        experiment_name=f"case_balanced_{replicate}",
        corpus_dir=CASE_BALANCED_WORK_ROOT / replicate,
        raw_runs_dir=CASE_BALANCED_WORK_ROOT / "runs",
        execution=case_execution(),
    )
    assert result["status"] == "completed", result["preview"]
    replicate_results[replicate] = result
    packaged = write_common_result_artifacts(
        result=result,
        category_map=category_map,
        category_order=category_order,
        support=support,
        bytes_by_category=bytes_by_category,
        output_dir=balanced_root / "replicates" / replicate,
        support_column="support",
        support_label="Distinct cases per category (log scale)",
        title_prefix=f"Case-balanced {replicate}",
        manifest=manifest,
    )
    replicate_tables.append(packaged)


distance: 100%|██████████| 435/435 [00:00<00:00, 1292.29pair/s]
/Users/erickpatrickbarcelos/codes/data-mining/venv/lib/python3.14/site-packages/numpy/lib/_function_base_impl.py:3036: RuntimeWarning: invalid value encountered in divide
  c /= stddev[:, None]
/Users/erickpatrickbarcelos/codes/data-mining/venv/lib/python3.14/site-packages/numpy/lib/_function_base_impl.py:3037: RuntimeWarning: invalid value encountered in divide
  c /= stddev[None, :]
distance: 100%|██████████| 435/435 [00:00<00:00, 1431.52pair/s]
/Users/erickpatrickbarcelos/codes/data-mining/venv/lib/python3.14/site-packages/numpy/lib/_function_base_impl.py:3036: RuntimeWarning: invalid value encountered in divide
  c /= stddev[:, None]
/Users/erickpatrickbarcelos/codes/data-mining/venv/lib/python3.14/site-packages/numpy/lib/_function_base_impl.py:3037: RuntimeWarning: invalid value encountered in divide
  c /= stddev[None, :]
distance: 100%|██████████| 435/435 [00:00<00:00, 1454.17pair/s]
/Users/erickpatrickbarcelos/code

## Build the representative balanced result pack


In [3]:
distance_matrices = {
    replicate: packaged["distance"]
    for replicate, packaged in zip(
        replicate_results,
        replicate_tables,
    )
}
matrix_stack = np.stack([matrix.to_numpy() for matrix in distance_matrices.values()])
median_matrix = pd.DataFrame(
    np.median(matrix_stack, axis=0),
    index=category_order,
    columns=category_order,
)
median_vector = np.array([
    median_matrix.loc[left, right]
    for left, right in combinations(category_order, 2)
])
medoid_rows = []
for replicate, matrix in distance_matrices.items():
    vector = np.array([
        matrix.loc[left, right]
        for left, right in combinations(category_order, 2)
    ])
    medoid_rows.append({
        "replicate": replicate,
        "distance_to_median": float(np.linalg.norm(vector - median_vector)),
    })
medoid_table = pd.DataFrame(medoid_rows).sort_values(
    ["distance_to_median", "replicate"]
)
representative_replicate = medoid_table.iloc[0]["replicate"]
representative = replicate_tables[
    list(replicate_results).index(representative_replicate)
]

balanced_output = RESULTS_ROOT / "case_balanced"
balanced_output.mkdir(parents=True, exist_ok=True)
median_matrix.to_csv(balanced_output / "distance-matrix.csv")
representative["membership_by_category"].to_csv(
    balanced_output / "clusters.csv", index=False
)
(balanced_output / "tree.nwk").write_text(
    representative["tree_newick"], encoding="utf-8"
)
support = case_category_map.loc[
    case_category_map["regime"] == "case-balanced",
    ["category", "case_count"],
].drop_duplicates("category").set_index("category").loc[category_order].reset_index()
support = support.rename(columns={"case_count": "support"})
support.to_csv(balanced_output / "support.csv", index=False)
plot_support(
    support,
    "support",
    "Distinct cases per category (log scale)",
    balanced_output / "support.png",
    "Case-balanced: category support",
)
plot_ncd_heatmap(
    median_matrix,
    balanced_output / "ncd-heatmap.png",
    "Case-balanced: median NCD distances",
)
replicate_diagnostics = pd.concat(
    [pd.read_csv(balanced_root / "replicates" / replicate / "bias-diagnostics.csv").assign(replicate=replicate)
     for replicate in distance_matrices],
    ignore_index=True,
)
median_diagnostics = (
    replicate_diagnostics.groupby("diagnostic", as_index=False)["spearman_correlation"]
    .median()
)
median_diagnostics.to_csv(balanced_output / "bias-diagnostics.csv", index=False)
plot_bias_diagnostics(
    median_diagnostics,
    balanced_output / "bias-diagnostics.png",
    "Case-balanced: median NCD bias diagnostics",
)
render_tree_artifacts(
    representative["tree_newick"],
    representative["membership_by_category"],
    category_map,
    balanced_output,
    f"Case-balanced representative ({representative_replicate})",
)

medoid_table.to_csv(balanced_output / "replicate-medoid-selection.csv", index=False)


## Replica stability


In [4]:
all_category_pairs = set(combinations(category_order, 2))
balanced_memberships = {
    replicate: packaged["membership_by_category"]
    for replicate, packaged in zip(replicate_results, replicate_tables)
}
balanced_cluster_pairs = {
    replicate: same_cluster_pairs(membership, category_order)
    for replicate, membership in balanced_memberships.items()
}
stability_rows = []
for left_name, right_name in combinations(balanced_memberships, 2):
    left_pairs = balanced_cluster_pairs[left_name]
    right_pairs = balanced_cluster_pairs[right_name]
    stability_rows.append({
        "left": left_name,
        "right": right_name,
        "pairwise_cluster_agreement": len(all_category_pairs - (left_pairs ^ right_pairs))
        / len(all_category_pairs),
    })
stability = pd.DataFrame(stability_rows)
stability.to_csv(balanced_output / "replicate-stability.csv", index=False)
plot_cluster_stability(stability, balanced_output / "replicate-stability.png")
write_json(
    balanced_output / "run-summary.json",
    {
        "experiment": "case_balanced",
        "category_set_version": manifest["category_set_version"],
        "category_definition_hash": manifest["category_definition_hash"],
        "category_count": manifest["category_count"],
        "dimension_count": manifest["dimension_count"],
        "replicas": list(replicate_results),
        "representative_replicate": representative_replicate,
        "status": "completed",
    },
)
display(medoid_table)
display(stability)


,replicate,distance_to_median
4,replicate-005,0.210160
2,replicate-003,0.218528
1,replicate-002,0.221491
0,replicate-001,0.223544
3,replicate-004,0.223995


,left,right,pairwise_cluster_agreement
0,replicate-001,replicate-002,0.903448
1,replicate-001,replicate-003,0.926437
2,replicate-001,replicate-004,0.901149
3,replicate-001,replicate-005,0.914943
4,replicate-002,replicate-003,0.908046
5,replicate-002,replicate-004,0.914943
6,replicate-002,replicate-005,0.896552
7,replicate-003,replicate-004,0.905747
8,replicate-003,replicate-005,0.896552
9,replicate-004,replicate-005,0.921839
